# Data Transforming — 13/07/2026

Transformación del dataset **tras** la fase de *Data Cleaning*, partiendo de
`clean_dataset_13_07_2026.csv` y **sobrescribiendo ese mismo fichero** con la versión enriquecida.

> **Alcance (importante).** El resultado es el **dataset único compartido por los tres roles**
> (Marketing, Operaciones y Experiencia de Cliente): se lee `Data/clean_dataset_13_07_2026.csv`
> y se **sobrescribe** con todas sus columnas más las variables derivadas.
>
> Las transformaciones son **idempotentes** (seguras ante reejecución): antes de recalcular cada
> variable se comprueba si ya existe y las columnas derivadas se **sobrescriben**, no se duplican.

## Definición del proceso

*Transformar los datos para prepararlos para el modelado y el análisis avanzado:*
- Normalización o escalado
- Codificación de variables categóricas
- Generación de nuevas variables

## Tareas a realizar
1. **Reescalado de puntuaciones** (rating 1000→100, resto 100→10) — *Experiencia de Cliente*.
2. **Columna binaria** `rating_above_80` — *Experiencia de Cliente*.
3. **Columnas de ocupación** (a partir de `availability_30/60/90/365`) — *Operaciones / KPI*.
4. **`is_instant_bookable` → binario (0/1)** — *Operaciones*.
5. **`has_availability` → binario (0/1)** — *Operaciones* → *(ya resuelto en Data Cleaning; aquí se **verifica**)*.
6. **Categorización de `amenities_list`** en 7 bloques temáticos + flags 0/1 — *Marketing*.
7. **`logp = log(price_€)`** — *Marketing*.
8.  **Cálculo de la ocupación máxima y mínima por barrio** (30, 60, 90 y 365 días) — *Marketing*.

> **Nota sobre los puntos 3, 4 y 5:** se mantienen exactamente igual que en `data_transform_29_06`
> (occupancy, `is_instant_bookable` binario y `has_availability` binario). Antes de recalcular cada
> tarea se comprueba si ya viene resuelta de *Data Cleaning* y, si es así, se documenta con un comentario.

## Librerías

In [69]:
import pandas as pd
import numpy as np
from pathlib import Path
import collections
pd.set_option('display.max_columns', None)

## Importación del CSV limpio
Partimos del CSV limpio (`clean_dataset_13_07_2026.csv`) entregado por *Data Cleaning*.
Trabajamos siempre sobre una copia para no tocar el dataset original importado.

In [70]:
def encontrar_raiz_proyecto(nombre_carpeta='Equip_34'):
    '''Encuentra la carpeta raíz del proyecto subiendo desde el directorio actual.'''
    actual = Path.cwd()
    for carpeta in [actual] + list(actual.parents):
        if carpeta.name == nombre_carpeta:
            return carpeta
    raise FileNotFoundError(f"No se encontró la carpeta '{nombre_carpeta}' subiendo desde {actual}")

raiz_proyecto = encontrar_raiz_proyecto('Equip_34')
ruta = raiz_proyecto / 'Data' / 'clean_dataset_13_07_2026.csv'
print(f"Ruta resuelta: {ruta}")

df_original = pd.read_csv(ruta, parse_dates=['first_review_date', 'last_review_date', 'insert_date'])
for c in ['bathrooms', 'bedrooms', 'beds']:
    df_original[c] = df_original[c].astype('Int64')
df = df_original.copy()
print("Dimensiones de partida:", df.shape)

Ruta resuelta: c:\Users\lacal\Desktop\IT_ACADEMY\simulador\ProjecteData\Equip_34\Data\clean_dataset_13_07_2026.csv
Dimensiones de partida: (9507, 54)


In [71]:
df.dtypes

apartment_id                            int64
name                                   object
description                            object
host_id                                 int64
neighbourhood_name                     object
neighbourhood_district                 object
room_type                              object
accommodates                            int64
bathrooms                               Int64
bedrooms                                Int64
beds                                    Int64
amenities_list                         object
price_€                               float64
minimum_nights                          int64
maximum_nights                          int64
has_availability                       object
availability_30                         int64
availability_60                         int64
availability_90                         int64
availability_365                        int64
number_of_reviews                       int64
first_review_date              dat

Confirmamos que seguimos con los mismos registros y tipos que entregó *Data Cleaning*
(incluida `has_availability_numeric`, ya creada allí).

## 1. Reescalado de puntuaciones (rating 1000→100, resto 100→10)

Las reseñas llegan de *Cleaning* sin reescalar (`review_scores_rating` en 0–1000 y las
sub-puntuaciones en 0–100). Aquí se dividen entre 10 para dejar `rating` en 0–100 y el resto
en 0–10, respetando los nulos estructurales (alojamientos sin reseñas).

> Se incluye una **comprobación de idempotencia**: si el CSV ya viniera reescalado
> (rating ≤ 100), no se vuelve a dividir.

In [72]:
cols_10 = ['review_scores_accuracy', 'review_scores_cleanliness', 'review_scores_checkin',
           'review_scores_communication', 'review_scores_location', 'review_scores_value']

rating_max = df['review_scores_rating'].max()
print("review_scores_rating máx. antes:", rating_max)

if rating_max is not pd.NA and rating_max <= 100:
    # Comprobación: la tarea ya estaba hecha (no se repite el reescalado)
    print("Las puntuaciones YA venían reescaladas -> no se vuelve a dividir.")
else:
    df['review_scores_rating'] = df['review_scores_rating'] / 10
    df[cols_10] = df[cols_10] / 10
    print("Reescalado aplicado: rating 1000->100 y sub-puntuaciones 100->10.")

# Verificación: nada fuera de escala
assert df['review_scores_rating'].max() <= 100
assert df[cols_10].max().max() <= 10
print("Verificación de escala OK.")
df[['review_scores_rating'] + cols_10].describe().round(2)

review_scores_rating máx. antes: 100.0
Las puntuaciones YA venían reescaladas -> no se vuelve a dividir.
Verificación de escala OK.


,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value
count,6915.00,6907.00,6912.00,6903.00,6912.00,6902.00,6902.00
mean,91.90,9.45,9.31,9.62,9.62,9.54,9.13
std,9.29,0.95,1.01,0.83,0.84,0.77,1.00
min,20.00,2.00,2.00,2.00,2.00,2.00,2.00
25%,89.00,9.00,9.00,9.00,9.00,9.00,9.00
50%,94.00,10.00,10.00,10.00,10.00,10.00,9.00
75%,98.00,10.00,10.00,10.00,10.00,10.00,10.00
max,100.00,10.00,10.00,10.00,10.00,10.00,10.00


## 2. Columna binaria: `rating_above_80`

Facilita el análisis de *Experiencia de Cliente*. Se calcula **después** del reescalado
(sobre la escala 0–100) y respeta los nulos (no se rellenan).

In [73]:
df['rating_above_80'] = (df['review_scores_rating'] > 80).astype('boolean')
df.loc[df['review_scores_rating'].isna(), 'rating_above_80'] = pd.NA
print(df['rating_above_80'].value_counts(dropna=False))

rating_above_80
True     6110
<NA>     2592
False     805
Name: count, dtype: Int64


## 3. Columnas de ocupación y tasa de ocupación

*(Punto 3 de `data_transform_29_06`, mantenido igual.)* Tarea **no** realizada en *Cleaning*,
se calcula aquí. `occupancy` interesa a *Operaciones*; `occupancy_rate` al KPI (la más relevante
es la de 30 días). Se sustituyen los días disponibles por los días totales del periodo en el
denominador, según la decisión de negocio.

In [74]:
plazos = {'30': 30, '60': 60, '90': 90, '365': 365}

for suf, total in plazos.items():
    df[f'occupancy_{suf}'] = total - df[f'availability_{suf}']
    df[f'occupancy_rate_{suf}'] = (df[f'occupancy_{suf}'] / total * 100).round(2)

df[[f'occupancy_{s}' for s in plazos] + [f'occupancy_rate_{s}' for s in plazos]].head()

,occupancy_30,occupancy_60,occupancy_90,occupancy_365,occupancy_rate_30,occupancy_rate_60,occupancy_rate_90,occupancy_rate_365
0,28,32,32,32,93.33,53.33,35.56,8.77
1,30,60,90,365,100.00,100.00,100.00,100.00
2,2,2,2,277,6.67,3.33,2.22,75.89
3,30,55,55,330,100.00,91.67,61.11,90.41
4,25,55,85,161,83.33,91.67,94.44,44.11


## 4. `is_instant_bookable` → binario (0/1)

*(Punto 4 de `data_transform_29_06`, mantenido igual.)* Tarea **no** realizada en *Cleaning*.
Se conserva la columna original y se crea `is_instant_bookable_numeric` (1/0) respetando nulos.
Esta codificación 0/1 es la que entra al modelo de *Operaciones* como dummy.

In [75]:
# Comprobación: sólo se crea si no existe ya de una fase previa
if 'is_instant_bookable_numeric' in df.columns:
    print("Ya existía 'is_instant_bookable_numeric' -> se conserva.")
else:
    df['is_instant_bookable_numeric'] = df['is_instant_bookable'].map({'VERDADERO': 1, 'FALSO': 0})
    print("Creada 'is_instant_bookable_numeric'.")

print(df['is_instant_bookable_numeric'].value_counts(dropna=False))
print(df['is_instant_bookable'].value_counts(dropna=False))
df[['is_instant_bookable', 'is_instant_bookable_numeric']].head()

Ya existía 'is_instant_bookable_numeric' -> se conserva.
is_instant_bookable_numeric
1    5512
0    3995
Name: count, dtype: int64
is_instant_bookable
VERDADERO    5512
FALSO        3995
Name: count, dtype: int64


,is_instant_bookable,is_instant_bookable_numeric
0,FALSO,0
1,FALSO,0
2,FALSO,0
3,VERDADERO,1
4,FALSO,0


## 5. `has_availability` → binario (0/1)  ·  *ya resuelto en Data Cleaning*

*(Punto 5 de `data_transform_29_06`.)* **Comprobación:** esta tarea **ya se realiza en
`Data_Cleaning_06_07`**, que crea la columna `has_availability_numeric` (1/0, nulos intactos como
`<NA>`). Aquí **no se repite**: sólo se verifica que existe y es coherente con la columna original.
Si por cualquier motivo no existiera (p. ej. un `clean_dataset` antiguo), se crea con la misma lógica.

In [76]:
if 'has_availability_numeric' in df.columns:
    # TAREA YA REALIZADA EN DATA CLEANING -> se verifica, no se recalcula
    print("OK: 'has_availability_numeric' ya viene de Data Cleaning. Se verifica la coherencia.")
else:
    print("No estaba presente -> se crea con la misma lógica que en Cleaning.")
    df['has_availability_numeric'] = df['has_availability'].map({'VERDADERO': 1, 'FALSO': 0}).astype('Int64')

# Verificación de coherencia original <-> binaria
chk = (df.assign(_na=df['has_availability'].isna())
         .groupby(['has_availability', 'has_availability_numeric'], dropna=False)
         .size())
print(chk)
print("\nDistribución final:")
print(df['has_availability_numeric'].value_counts(dropna=False))

OK: 'has_availability_numeric' ya viene de Data Cleaning. Se verifica la coherencia.
has_availability  has_availability_numeric
VERDADERO         1.0                         8977
NaN               NaN                          530
dtype: int64

Distribución final:
has_availability_numeric
1.0    8977
NaN     530
Name: count, dtype: int64


## 6. Categorización de `amenities_list` en 7 bloques temáticos (Marketing)

La columna `amenities_list` ya viene **limpia y normalizada** de *Data Cleaning* (formato unificado,
sin `translation missing`, sin corchetes sueltos, sinónimos y mojibake consolidados → 231 comodidades
únicas). Aquí se **agrupan en 7 categorías de negocio** y se generan **flags binarios `cat_*` (0/1)**,
que son la variable que entra al modelo (no la lista de texto).

**Método (opción de mayor precisión):** *mapeo curado* comodidad→categoría construido a partir de la
lista maestra (`frecuencia_amenities.csv`). Cada comodidad se asigna explícitamente a **una única**
categoría. Se añade una **red de seguridad por palabras clave** para clasificar cualquier variante no
listada, y se imprime una **auditoría** de qué comodidad cae en cada bloque (y cuáles quedan sin
categoría a propósito: básicas casi universales como *Essentials/Hangers/Iron/Washer…* y accesibilidad,
que no forman parte de los 7 bloques y diluirían la señal).

In [77]:
# --- Parseo de amenities_list a lista por registro (ya viene limpia de Data Cleaning) ---
def parse_amenities(v):
    if pd.isna(v):
        return []
    return [a.strip() for a in str(v).split(',') if a.strip()]

df['amenities_parsed'] = df['amenities_list'].apply(parse_amenities)

# --- MAPEO CURADO comodidad -> categoría (una categoría por comodidad) ---
CATEGORIAS = {
 'premium': {
    'Pool', 'Pool with pool hoist', 'Pool toys', 'Shared pool', 'Shared outdoor pool',
    'Hot tub', 'Indoor fireplace', 'Gym', 'Exercise equipment',
    'BBQ grill', 'Barbecue utensils',
    'Waterfront', 'Beachfront', 'Beach view', 'Beach essentials', 'Lake access', 'Mountain view',
    'Garden or backyard', 'Shared garden or backyard',
    'Bathtub', 'Soaking tub', 'Bathtub with bath chair', 'Rain shower',
    'Doorman', 'Building staff', 'Front desk/doorperson', 'Standing valet',
    'Terrace', 'Balcony', 'Patio or balcony', 'Outdoor seating', 'Outdoor furniture',
    'Outdoor dining area', 'Sun loungers', 'Hammock', 'Outdoor shower',
    'Ski-in/Ski-out', 'EV charger',
 },
 'confort': {
    'Air conditioning', 'Central air conditioning',
    'Heating', 'Central heating', 'Portable heater',
    'Portable fans', 'Ceiling fan', 'Heated floors', 'Heated towel rack',
    'Dishwasher', 'Dryer', 'Elevator', 'Elevator in building',
 },
 'conectividad': {
    'Wifi', 'Pocket wifi', 'Internet', 'Ethernet connection',
    'TV', 'Cable TV', 'Smart TV', '40 HDTV', '43 HDTV with Netflix', 'Netflix', 'HBO GO',
    'DVD player', 'Game console',
    'Laptop friendly workspace', 'Dedicated workspace', 'Office',
    'Sound system', 'Bluetooth sound system', 'IKEA NEARBY Bluetooth sound system',
    'Amazon Echo', 'Printer',
 },
 'familia': {
    'Family/kid friendly', 'Crib', 'High chair', "Pack 'n Play/travel crib",
    "Children's books and toys", "Children's dinnerware",
    'Baby bath', 'Baby monitor', 'Baby safety gates', 'Babysitter recommendations',
    'Changing table', 'Stair gates', 'Table corner guards', 'Outlet covers',
    'Window guards', 'Fireplace guards',
 },
 'seguridad': {
    'Smoke alarm', 'Carbon monoxide alarm', 'Fire extinguisher', 'First aid kit', 'Safety card',
    'Lock on bedroom door', 'Security system', 'Smart lock', 'Keypad', 'Lockbox',
 },
 'parking': {
    'Free parking on premises', 'Free street parking', 'Free parking on street',
    'Paid parking off premises', 'Paid parking on premises',
    'Paid parking garage off premises', 'Paid parking garage on premises',
    'Free driveway parking on premises - 1 space', 'Disabled parking spot', 'Parking',
 },
 'cocina': {
    'Kitchen', 'Kitchenette', 'Full kitchen', "Chef's kitchen",
    'Oven', 'Double oven', 'Gas oven', 'Convection oven', 'Steam oven', 'Stainless steel oven',
    'Warming drawer', 'Stove', 'Electric stove', 'Stainless steel stove',
    'Stainless steel electric stove',
    'Microwave', 'Refrigerator', 'Mini fridge', 'Freezer',
    'Coffee maker', 'Espresso machine', 'Nespresso machine', 'Pour-over coffee', 'Hot water kettle',
    'Cooking basics', 'Dishes and silverware', 'Toaster', 'Rice Maker', 'Bread maker', 'Baking sheet',
    'Wine glasses', 'Dining table', 'Dining area', 'Formal dining area', 'Breakfast table', 'Breakfast',
 },
}
CATS = list(CATEGORIAS.keys())
CAT_LOWER = {cat: {a.lower() for a in ams} for cat, ams in CATEGORIAS.items()}

# Red de seguridad por palabras clave (sólo para comodidades NO listadas explícitamente)
KEYWORDS = {
 'premium': ['pool', 'hot tub', 'jacuzzi', 'sauna', 'fireplace', 'bbq', 'grill', 'gym',
             'waterfront', 'beachfront', 'beach ', 'garden', 'backyard', 'bathtub',
             'doorman', 'building staff', 'terrace', 'balcony', 'patio', 'ski-in',
             'sun lounger', 'hammock'],
 # OJO: no se incluye 'dryer' como keyword porque captaría "Hair dryer" (básico) además de
 # "Dryer" (secadora); "Dryer" ya está en el set explícito, así que "Hair dryer" queda sin categoría.
 'confort': ['air conditioning', 'heating', 'dishwasher', 'elevator', 'fan', 'heated'],
 'conectividad': ['wifi', 'internet', 'ethernet', 'tv', 'netflix', 'hdtv', 'workspace',
                  'laptop', 'sound system'],
 'familia': ['crib', 'high chair', 'family', 'children', 'pack ', 'baby', 'changing table',
             'stair gate', 'child'],
 'seguridad': ['smoke', 'carbon monoxide', 'fire extinguisher', 'first aid', 'safety card',
               'lock on bedroom', 'security', 'smart lock', 'keypad', 'lockbox'],
 'parking': ['parking'],
 'cocina': ['kitchen', 'oven', 'stove', 'microwave', 'refrigerator', 'fridge', 'freezer',
            'coffee', 'espresso', 'nespresso', 'cooking', 'dishes', 'toaster', 'kettle',
            'dining', 'breakfast'],
}

def clasificar(amenity):
    a = amenity.lower()
    for cat, s in CAT_LOWER.items():   # 1) mapeo curado explícito
        if a in s:
            return cat
    for cat, kws in KEYWORDS.items():  # 2) red de seguridad por keyword
        if any(k in a for k in kws):
            return cat
    return None

### 6.1 Auditoría del mapeo (trazabilidad)
Se lista, para cada categoría, qué comodidades caen dentro y cuáles quedan **sin categoría**
(a propósito). Esto documenta y hace auditable la decisión de negocio.

In [78]:
todas = sorted({a for L in df['amenities_parsed'] for a in L})
asignacion = {a: clasificar(a) for a in todas}

por_cat = collections.defaultdict(list)
for a, c in asignacion.items():
    por_cat[c].append(a)

for cat in CATS:
    ams = sorted(por_cat[cat])
    print(f"[{cat}]  ({len(ams)} comodidades)")
    print("   " + ", ".join(ams))
    print()

sin_cat = sorted(por_cat[None])
print(f"[SIN CATEGORÍA — no forman parte de los 7 bloques]  ({len(sin_cat)} comodidades)")
print("   " + ", ".join(sin_cat))

[premium]  (40 comodidades)
   BBQ grill, Balcony, Barbecue utensils, Bathtub, Bathtub with bath chair, Beach essentials, Beach view, Beachfront, Building staff, Doorman, EV charger, Exercise equipment, Front desk/doorperson, Garden or backyard, Gym, Hammock, Hot tub, Indoor fireplace, Lake access, Mountain view, Outdoor dining area, Outdoor furniture, Outdoor seating, Outdoor shower, Patio or balcony, Pool, Pool toys, Pool with pool hoist, Rain shower, Saltwater pool, Shared garden or backyard, Shared outdoor pool, Shared pool, Ski-in/Ski-out, Soaking tub, Standing valet, Sun loungers, Terrace, Waterfront, Wood-burning fireplace

[confort]  (15 comodidades)
   Air conditioning, Ceiling fan, Central air conditioning, Central heating, Dishwasher, Dryer, Elevator, Elevator in building, Heated floors, Heated towel rack, Heating, Portable air conditioning, Portable fans, Portable heater, Radiant heating

[conectividad]  (23 comodidades)
   40 HDTV, 43 HDTV with Netflix, Amazon Echo, Blueto

### 6.2 Flags binarios `cat_*`
`cat_x = 1` si el alojamiento tiene **al menos una** comodidad del bloque `x`.

In [79]:
def flags_row(L):
    cats = set()
    for a in L:
        c = clasificar(a)
        if c is not None:
            cats.add(c)
    return pd.Series({f'cat_{c}': int(c in cats) for c in CATS})

flags = df['amenities_parsed'].apply(flags_row)
catcols = [f'cat_{c}' for c in CATS]
for col in catcols:                 # asignación (sobrescribe si ya existiera -> seguro ante reejecución)
    df[col] = flags[col]

print("Proporción de alojamientos con cada categoría:")
print(df[catcols].mean().round(3))

Proporción de alojamientos con cada categoría:
cat_premium         0.517
cat_confort         0.905
cat_conectividad    0.985
cat_familia         0.540
cat_seguridad       0.519
cat_parking         0.507
cat_cocina          0.955
dtype: float64


## 7. Transformación logarítmica de `price_€` (Marketing)

Se crea `logp = log(price_€)` dado el fuerte sesgo a la derecha del precio (skew ≈ 1,76 → ≈ 0,09
tras el log). Convierte los coeficientes del modelo en **semi-elasticidades** (lectura en "+X %").
Los nulos de `price_€` se mantienen como `NaN` (no se eliminan filas aquí: se hace en el análisis).

In [80]:
df['logp'] = np.log(df['price_€'])
print("skew price_€:", round(df['price_€'].skew(), 2), "-> skew logp:", round(df['logp'].skew(), 2))
print("Nulos en logp (= nulos en price_€, no se eliminan aquí):", int(df['logp'].isna().sum()))
df[['price_€', 'logp']].describe().round(2)

skew price_€: 7.18 -> skew logp: 0.11
Nulos en logp (= nulos en price_€, no se eliminan aquí): 239


,price_€,logp
count,9268.00,9268.00
mean,1021.23,6.63
std,981.85,0.77
min,60.00,4.09
25%,450.00,6.11
50%,750.00,6.62
75%,1240.00,7.12
max,28571.00,10.26


## 8.  **Cálculo de la ocupación máxima y mínima por barrio** (30, 60, 90 y 365 días) — *Marketing*. 

In [81]:
ocupacion_barrio = (
    df.groupby("neighbourhood_name")
      .agg({
          "occupancy_rate_30": ["min", "max"],
          "occupancy_rate_60": ["min", "max"],
          "occupancy_rate_90": ["min", "max"],
          "occupancy_rate_365": ["min", "max"]
      })
)

ocupacion_barrio

occupancy_rate_30         occupancy_rate_60          \
                                 min     max               min     max   
neighbourhood_name                                                       
AIORA                           0.00  100.00              0.00  100.00   
ALBORS                         70.00  100.00             76.67  100.00   
ARRANCAPINS                     3.33  100.00              5.00  100.00   
Abrantes                       46.67  100.00             23.33  100.00   
Acacias                         0.00  100.00              0.00  100.00   
...                              ...     ...               ...     ...   
la Vila de Gr�cia               0.00  100.00              0.00  100.00   
les Corts                       0.00  100.00              0.00  100.00   
les Roquetes                    0.00   70.00              0.00   61.67   
les Tres Torres                 0.00  100.00              0.00  100.00   
no asignado                     0.00   66.67              0.00   50.00   

                   occupancy_rate_90         occupancy_rate_365          
                                 min     max                min     max  
neighbourhood_name                                                       
AIORA                           0.00  100.00               8.49  100.00  
ALBORS                         81.11  100.00              32.05  100.00  
ARRANCAPINS                     3.33   67.78               2.74   75.89  
Abrantes                       15.56  100.00              79.18  100.00  
Acacias                         0.00  100.00               0.00  100.00  
...                              ...     ...                ...     ...  
la Vila de Gr�cia               0.00  100.00               0.00  100.00  
les Corts                       0.00  100.00               1.10  100.00  
les Roquetes                    0.00   54.44              26.30   87.67  
les Tres Torres                 0.00  100.00               0.00  100.00  
no asignado                     0.00   42.22              41.92   50.68  

[520 rows x 8 columns]

## 9. Verificación final y exportación

Se elimina sólo la columna auxiliar `amenities_parsed` (lista de Python, no serializable a CSV) y se
**sobrescribe `Data/clean_dataset_13_07_2026.csv`** con la versión enriquecida. **No se ha eliminado
ninguna fila**: el dataset conserva los 7.693 registros y todas las columnas del CSV limpio más las
derivadas.

In [82]:
df_export = df.drop(columns=['amenities_parsed'])

print("Dimensiones finales:", df_export.shape)
print("apartment_id único:", df_export['apartment_id'].is_unique)
print("Filas (deben seguir siendo 7693):", len(df_export))

# Comprobación de columnas requeridas por los análisis
requeridas = ['apartment_id', 'city', 'insert_date',
              'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness',
              'review_scores_checkin', 'review_scores_communication',
              'price_€', 'accommodates', 'number_of_reviews', 'room_type',
              'is_instant_bookable_numeric', 'has_availability_numeric',
              'occupancy_30', 'occupancy_60', 'occupancy_90', 'occupancy_365',
              'logp'] + catcols
faltan = [c for c in requeridas if c not in df_export.columns]
print("Columnas requeridas que faltan:", faltan if faltan else "ninguna ✓")
print("\nTotal de columnas:", len(df_export.columns))
list(df_export.columns)

Dimensiones finales: (9507, 54)
apartment_id único: True
Filas (deben seguir siendo 7693): 9507
Columnas requeridas que faltan: ninguna ✓

Total de columnas: 54


['apartment_id',
 'name',
 'description',
 'host_id',
 'neighbourhood_name',
 'neighbourhood_district',
 'room_type',
 'accommodates',
 'bathrooms',
 'bedrooms',
 'beds',
 'amenities_list',
 'price_€',
 'minimum_nights',
 'maximum_nights',
 'has_availability',
 'availability_30',
 'availability_60',
 'availability_90',
 'availability_365',
 'number_of_reviews',
 'first_review_date',
 'last_review_date',
 'review_scores_rating',
 'review_scores_accuracy',
 'review_scores_cleanliness',
 'review_scores_checkin',
 'review_scores_communication',
 'review_scores_location',
 'review_scores_value',
 'is_instant_bookable',
 'reviews_per_month',
 'country',
 'city',
 'insert_date',
 'has_availability_numeric',
 'rating_above_80',
 'occupancy_30',
 'occupancy_rate_30',
 'occupancy_60',
 'occupancy_rate_60',
 'occupancy_90',
 'occupancy_rate_90',
 'occupancy_365',
 'occupancy_rate_365',
 'is_instant_bookable_numeric',
 'cat_premium',
 'cat_confort',
 'cat_conectividad',
 'cat_familia',
 'cat_segur

In [83]:
# Se sobrescribe el mismo CSV limpio con la versión enriquecida (dataset único para todos los roles)
ruta_salida = raiz_proyecto / 'Data' / 'clean_dataset_13_07_2026.csv'
df_export.to_csv(ruta_salida, index=False, encoding='utf-8')
print("clean_dataset_13_07_2026.csv sobrescrito en:", ruta_salida.resolve())
print("Filas x columnas:", df_export.shape)

clean_dataset_13_07_2026.csv sobrescrito en: C:\Users\lacal\Desktop\IT_ACADEMY\simulador\ProjecteData\Equip_34\Data\clean_dataset_13_07_2026.csv
Filas x columnas: (9507, 54)
